In [ ]:
!pip install pennylane pennylane-lightning[gpu] numpy scipy custatevec_cu12 dataclasses

# Single-Shot Quantum mtDNA Vectorization and Contextual Error-Correction Audit

This notebook evaluates the same 100 synthetic mtDNA inputs with a 12-qubit QFT circuit, but the **measurement layer is strict single-shot**.

The scientific separation used throughout the notebook is:

$$
\text{legitimate data/circuit evolution}
\neq
\text{contextual injected error}
\neq
\text{single-shot sampling deviation}.
$$

For a chunk $x$ the independently propagated clean state is

$$
|\psi^{\mathrm{ideal}}(x)\rangle
=
U_{\mathrm{QFT}}|\psi_{\mathrm{in}}(x)\rangle.
$$

The noisy trajectory applies an error operator after each QFT operation:

$$
|\psi^{\mathrm{noisy}}_L(x)\rangle
=
E_L U_L \cdots E_2 U_2 E_1 U_1 |\psi_{\mathrm{in}}(x)\rangle.
$$

Therefore an error introduced early in the circuit is acted on by all later QFT gates. It is **not** appended only at the output.

The mtDNA encoding and every legitimate QFT operation are included in the clean contextual reference. Consequently, changes caused by the input sequence or by legitimate QFT evolution are not classified as errors.

The exact clean/noisy states are retained only to construct independently propagated **expected values and audit quantities**. Actual measurement diagnostics and kernel observations use exactly one Bernoulli/categorical draw per measurement.

In [ ]:
# =====================================================================
# QUANTUM mtDNA VECTORIZATION + CONTEXTUAL ERROR CORRECTION
# 100 synthetic individuals | <= 16,655 bp
# 12 qubits | 4096 amplitudes/chunk | QFT | 100x100 kernel
# =====================================================================

import numpy as np
import pennylane as qml
from dataclasses import dataclass

# =====================================================================
# 1. CONFIGURATION
# =====================================================================

SEED = 42
rng = np.random.default_rng(SEED)

NUM_PEOPLE = 100
MTDNA_LENGTH = 16655

N_QUBITS = 12
CHUNK_SIZE = 2 ** N_QUBITS            # 4096
N_CHUNKS = int(np.ceil(MTDNA_LENGTH / CHUNK_SIZE))  # 5

# Synthetic variation
MIN_MUTATIONS = 20
MAX_MUTATIONS = 120

# Contextual coherent error ranges
NOISE_SEED = SEED + 15015

RX_ERROR_RANGE = (-0.060, 0.060)
RY_ERROR_RANGE = (-0.080, 0.080)
RZ_ERROR_RANGE = (-0.060, 0.060)

EPS = 1e-12

print("=" * 90)
print("CONFIGURATION")
print("=" * 90)
print(f"Individuals        : {NUM_PEOPLE}")
print(f"mtDNA length       : {MTDNA_LENGTH:,} bp")
print(f"Qubits             : {N_QUBITS}")
print(f"Hilbert dimension  : {CHUNK_SIZE:,}")
print(f"Chunks/person      : {N_CHUNKS}")
print(f"Maximum bases used : {N_CHUNKS * CHUNK_SIZE:,} including final padding")
print()

CONFIGURATION
Individuals        : 100
mtDNA length       : 16,655 bp
Qubits             : 12
Hilbert dimension  : 4,096
Chunks/person      : 5
Maximum bases used : 20,480 including final padding



In [ ]:
# =====================================================================
# STRICT SINGLE-SHOT CONFIGURATION
# =====================================================================
SHOTS = 1
SHOT_SEED = SEED + 8080
shot_rng = np.random.default_rng(SHOT_SEED)

print("=" * 90)
print("STRICT SINGLE-SHOT CONFIGURATION")
print("=" * 90)
print(f"Shots per measurement : {SHOTS}")
print(f"Shot RNG seed         : {SHOT_SEED}")
print("Exact states are used only for independent expected-reference/audit branches.")
print()

STRICT SINGLE-SHOT CONFIGURATION
Shots per measurement : 1
Shot RNG seed         : 8122
Exact states are used only for independent expected-reference/audit branches.



In [ ]:
# =====================================================================
# 2. SYNTHETIC mtDNA DATASET
# =====================================================================

BASES = np.array(["A", "C", "G", "T"])


def generate_reference_mtdna(length=MTDNA_LENGTH):
    """
    Synthetic reference mitochondrial sequence.
    Length never exceeds MTDNA_LENGTH.
    """
    return rng.choice(BASES, size=length)


def mutate_sequence(reference, n_mutations):
    """
    Introduces substitutions only.
    Sequence length therefore remains exactly constant.
    """
    seq = reference.copy()

    positions = rng.choice(
        len(seq),
        size=min(n_mutations, len(seq)),
        replace=False
    )

    for pos in positions:
        original = seq[pos]
        alternatives = BASES[BASES != original]
        seq[pos] = rng.choice(alternatives)

    return seq, positions


reference_mtdna = generate_reference_mtdna()

dataset = []
mutation_records = []

for person_id in range(NUM_PEOPLE):

    if person_id == 0:
        sequence = reference_mtdna.copy()
        positions = np.array([], dtype=np.int32)
    else:
        n_mut = int(
            rng.integers(
                MIN_MUTATIONS,
                MAX_MUTATIONS + 1
            )
        )

        sequence, positions = mutate_sequence(
            reference_mtdna,
            n_mut
        )

    assert len(sequence) <= MTDNA_LENGTH

    dataset.append(sequence)
    mutation_records.append(positions)

print("=" * 90)
print("SYNTHETIC mtDNA DATASET")
print("=" * 90)
print(f"Generated individuals : {len(dataset)}")
print(f"Sequence length        : {len(dataset[0]):,} bp")
print(f"Maximum allowed        : {MTDNA_LENGTH:,} bp")
print(f"Person 000 mutations   : {len(mutation_records[0])}")
print(f"Person 001 mutations   : {len(mutation_records[1])}")
print()


# =====================================================================
# 3. NUCLEOTIDE ENCODING
#
# Avoid ASCII geometry.
#
# A -> +1.0
# C -> +0.5
# G -> -0.5
# T -> -1.0
#
# Padding -> 0.0
# =====================================================================

NUCLEOTIDE_MAP = {
    "A":  1.0,
    "C":  0.5,
    "G": -0.5,
    "T": -1.0,
}


def encode_sequence(sequence):

    encoded = np.fromiter(
        (NUCLEOTIDE_MAP[b] for b in sequence),
        dtype=np.float32,
        count=len(sequence)
    )

    assert encoded.size <= MTDNA_LENGTH

    return encoded


def split_into_chunks(encoded):

    chunks = []
    valid_lengths = []

    for start in range(0, len(encoded), CHUNK_SIZE):

        chunk = encoded[start:start + CHUNK_SIZE]

        valid_len = len(chunk)

        padded = np.zeros(
            CHUNK_SIZE,
            dtype=np.float32
        )

        padded[:valid_len] = chunk

        # AmplitudeEmbedding cannot normalize an all-zero vector.
        norm = np.linalg.norm(padded)

        if norm < EPS:
            padded[0] = 1.0

        chunks.append(padded)
        valid_lengths.append(valid_len)

    return (
        np.asarray(chunks, dtype=np.float32),
        np.asarray(valid_lengths, dtype=np.int32)
    )

SYNTHETIC mtDNA DATASET
Generated individuals : 100
Sequence length        : 16,655 bp
Maximum allowed        : 16,655 bp
Person 000 mutations   : 0
Person 001 mutations   : 77



## Explicit QFT and contextual error injection

The QFT is decomposed gate-by-gate rather than treated as an opaque block. For an input basis component $|x\rangle$,

$$
U_{\mathrm{QFT}}|x\rangle
=
\frac{1}{\sqrt{2^n}}
\sum_{k=0}^{2^n-1}
e^{2\pi i xk/2^n}|k\rangle.
$$

The clean and noisy branches receive the same encoded mtDNA chunk and the same legitimate QFT gates. Only the noisy branch receives contextual error rotations after each QFT operation. This makes the same-depth clean trajectory the expected contextual reference.

In [ ]:
# =====================================================================
# 4. EXPLICIT QFT DECOMPOSITION
#
# We do NOT call qml.QFT as one opaque block.
#
# This lets us inject errors INSIDE the QFT so that an error introduced
# at one operation propagates through all subsequent operations.
# =====================================================================

@dataclass
class QFTOperation:
    kind: str
    wires: tuple
    angle: float = 0.0


def build_qft_operations(n_qubits):

    operations = []

    for target in range(n_qubits):

        operations.append(
            QFTOperation(
                kind="H",
                wires=(target,)
            )
        )

        for control in range(target + 1, n_qubits):

            angle = np.pi / (2 ** (control - target))

            operations.append(
                QFTOperation(
                    kind="CPHASE",
                    wires=(control, target),
                    angle=angle
                )
            )

    # Reverse qubit order
    for i in range(n_qubits // 2):

        operations.append(
            QFTOperation(
                kind="SWAP",
                wires=(i, n_qubits - i - 1)
            )
        )

    return operations


QFT_OPERATIONS = build_qft_operations(N_QUBITS)

print("=" * 90)
print("QFT STRUCTURE")
print("=" * 90)
print(f"QFT operations : {len(QFT_OPERATIONS)}")
print()

QFT STRUCTURE
QFT operations : 84



In [ ]:
# =====================================================================
# 5. CONTEXTUAL HARDWARE ERROR MODEL
#
# BASE_ERROR_MAP models a persistent gate/qubit-dependent coherent bias.
# Each person/chunk is evaluated in a slightly different execution context
# (deterministic calibration drift). This is important: if every sample sees
# exactly the same unitary error U, fidelity kernels are invariant because
# <Ui|Uj> = <i|j>. Context drift breaks that artificial common-unitary symmetry
# while preserving gate-by-gate error propagation through QFT depth.
# =====================================================================

noise_rng = np.random.default_rng(NOISE_SEED)

BASE_ERROR_MAP = np.zeros(
    (len(QFT_OPERATIONS), N_QUBITS, 3),
    dtype=np.float64
)

BASE_ERROR_MAP[:, :, 0] = noise_rng.uniform(
    RX_ERROR_RANGE[0], RX_ERROR_RANGE[1],
    size=(len(QFT_OPERATIONS), N_QUBITS)
)
BASE_ERROR_MAP[:, :, 1] = noise_rng.uniform(
    RY_ERROR_RANGE[0], RY_ERROR_RANGE[1],
    size=(len(QFT_OPERATIONS), N_QUBITS)
)
BASE_ERROR_MAP[:, :, 2] = noise_rng.uniform(
    RZ_ERROR_RANGE[0], RZ_ERROR_RANGE[1],
    size=(len(QFT_OPERATIONS), N_QUBITS)
)

# Small run-to-run / execution-context drift around the persistent map.
# These are deliberately smaller than the base coherent-error ranges.
DRIFT_RX = 0.015
DRIFT_RY = 0.020
DRIFT_RZ = 0.015


def contextual_error_map(person_idx, chunk_idx):
    """Deterministic execution-context map for one person/chunk.

    Same seed + same person/chunk => exactly reproducible map.
    Errors remain gate-, qubit- and axis-dependent and are injected after
    every QFT operation, so early errors propagate through later QFT gates.
    """
    local_rng = np.random.default_rng(
        np.random.SeedSequence([NOISE_SEED, int(person_idx), int(chunk_idx)])
    )

    drift = np.empty_like(BASE_ERROR_MAP)
    drift[:, :, 0] = local_rng.uniform(-DRIFT_RX, DRIFT_RX, BASE_ERROR_MAP.shape[:2])
    drift[:, :, 1] = local_rng.uniform(-DRIFT_RY, DRIFT_RY, BASE_ERROR_MAP.shape[:2])
    drift[:, :, 2] = local_rng.uniform(-DRIFT_RZ, DRIFT_RZ, BASE_ERROR_MAP.shape[:2])

    return BASE_ERROR_MAP + drift


print("=" * 90)
print("CONTEXTUAL HARDWARE ERROR MODEL")
print("=" * 90)
print(f"Noise seed       : {NOISE_SEED}")
print(f"Base RX range    : [{BASE_ERROR_MAP[:,:,0].min():+.5f}, {BASE_ERROR_MAP[:,:,0].max():+.5f}]")
print(f"Base RY range    : [{BASE_ERROR_MAP[:,:,1].min():+.5f}, {BASE_ERROR_MAP[:,:,1].max():+.5f}]")
print(f"Base RZ range    : [{BASE_ERROR_MAP[:,:,2].min():+.5f}, {BASE_ERROR_MAP[:,:,2].max():+.5f}]")
print(f"Context drift RX : ±{DRIFT_RX:.5f}")
print(f"Context drift RY : ±{DRIFT_RY:.5f}")
print(f"Context drift RZ : ±{DRIFT_RZ:.5f}")
print()


CONTEXTUAL HARDWARE ERROR MODEL
Noise seed       : 15057
Base RX range    : [-0.05988, +0.05998]
Base RY range    : [-0.07980, +0.07990]
Base RZ range    : [-0.05988, +0.05998]
Context drift RX : ±0.01500
Context drift RY : ±0.02000
Context drift RZ : ±0.01500



In [ ]:
# =====================================================================
# 6. QUANTUM DEVICES
# =====================================================================

try:
    dev_clean = qml.device("lightning.gpu", wires=N_QUBITS)
    dev_noisy = qml.device("lightning.gpu", wires=N_QUBITS)
    BACKEND = "lightning.gpu"
except Exception:
    dev_clean = qml.device("lightning.qubit", wires=N_QUBITS)
    dev_noisy = qml.device("lightning.qubit", wires=N_QUBITS)
    BACKEND = "lightning.qubit"

print(f"Quantum backend : {BACKEND}")
print()


# =====================================================================
# 7. QFT GATE APPLICATION
# =====================================================================

def apply_qft_operation(op):
    if op.kind == "H":
        qml.Hadamard(wires=op.wires[0])
    elif op.kind == "CPHASE":
        qml.ControlledPhaseShift(op.angle, wires=list(op.wires))
    elif op.kind == "SWAP":
        qml.SWAP(wires=list(op.wires))


def inject_contextual_error(operation_index, error_map):
    """Inject the context-specific error after one QFT operation.

    The resulting corrupted state is then passed through every remaining
    QFT operation, preserving contextual propagation through circuit depth.
    """
    for q in range(N_QUBITS):
        ex = error_map[operation_index, q, 0]
        ey = error_map[operation_index, q, 1]
        ez = error_map[operation_index, q, 2]
        qml.RX(ex, wires=q)
        qml.RY(ey, wires=q)
        qml.RZ(ez, wires=q)


# =====================================================================
# 8. CLEAN QFT
# =====================================================================

@qml.qnode(dev_clean)
def clean_qft_state(vector):
    qml.AmplitudeEmbedding(
        features=vector,
        wires=range(N_QUBITS),
        normalize=True
    )
    for op in QFT_OPERATIONS:
        apply_qft_operation(op)
    return qml.state()


# =====================================================================
# 9. CONTEXTUAL NOISY QFT
# =====================================================================

@qml.qnode(dev_noisy)
def noisy_qft_state(vector, error_map):
    qml.AmplitudeEmbedding(
        features=vector,
        wires=range(N_QUBITS),
        normalize=True
    )

    for operation_index, op in enumerate(QFT_OPERATIONS):
        apply_qft_operation(op)
        inject_contextual_error(operation_index, error_map)

    return qml.state()


# =====================================================================
# 10. STATE NORMALIZATION / PHASE ALIGNMENT
# =====================================================================

def normalize_state(state):
    state = np.asarray(state, dtype=np.complex128)
    norm = np.linalg.norm(state)
    if norm < EPS:
        raise ValueError("Zero quantum state encountered.")
    return state / norm


def align_global_phase(reference, state):
    """Remove physically irrelevant global phase before residual audit."""
    overlap = np.vdot(reference, state)
    if np.abs(overlap) < EPS:
        return state
    phase = overlap / np.abs(overlap)
    return state * np.conj(phase)


# =====================================================================
# 11. DETERMINISTIC CONTEXTUAL REFERENCE CORRECTION
#
# residual  = noisy - expected
# corrected = noisy - residual = expected
# =====================================================================

def contextual_correct_state(noisy_state, expected_state):
    noisy_state = normalize_state(noisy_state)
    expected_state = normalize_state(expected_state)

    noisy_aligned = align_global_phase(expected_state, noisy_state)
    residual = noisy_aligned - expected_state
    corrected = noisy_aligned - residual
    corrected = normalize_state(corrected)

    return noisy_aligned, corrected, residual


Quantum backend : lightning.gpu



## Expected trajectories, one-shot observation, and leakage boundary

For computational-basis state $z$, define the same-depth ideal and noisy expected probabilities

$$
E^{\mathrm{ideal}}(z)=|\langle z|\psi^{\mathrm{ideal}}\rangle|^2,
$$

$$
E^{\mathrm{noisy}}(z)=|\langle z|\psi^{\mathrm{noisy}}\rangle|^2.
$$

A strict one-shot measurement produces a one-hot vector $M^{(1)}$. The total residual relative to the contextual ideal reference is

$$
R(z)=M^{(1)}(z)-E^{\mathrm{ideal}}(z).
$$

For audit only, this can be decomposed exactly as

$$
R(z)
=
\underbrace{
E^{\mathrm{noisy}}(z)-E^{\mathrm{ideal}}(z)
}_{\Delta_{\mathrm{context}}(z)}
+
\underbrace{
M^{(1)}(z)-E^{\mathrm{noisy}}(z)
}_{S^{(1)}(z)}.
$$

The correction function is deliberately isolated. Its only inputs are

$$
M^{(1)}
\quad\text{and}\quad
E^{\mathrm{ideal}}.
$$

It does **not** receive `BASE_ERROR_MAP`, the person/chunk contextual error map, injected $R_X/R_Y/R_Z$ angles, `NOISE_SEED`, or $E^{\mathrm{noisy}}$.

Thus there is no direct injection-parameter leakage into correction. The method is nevertheless a **reference-based deterministic projection**, because the independently propagated ideal contextual expectation is intentionally available:

$$
M^{\mathrm{corr}}
=
M^{(1)}
-
\left(
M^{(1)}-E^{\mathrm{ideal}}
\right)
=
E^{\mathrm{ideal}}.
$$

Accordingly, 100% reference recovery is an algebraic consequence of this correction definition and must not be interpreted as blind recovery of an unknown physical hardware error from one shot.

In [ ]:
# =====================================================================
# SINGLE-SHOT HELPERS + VECTORIZATION
# =====================================================================

def state_probabilities(state):
    state = normalize_state(state)
    p = np.abs(state) ** 2
    p = np.asarray(p, dtype=np.float64)
    return p / p.sum()

def one_shot_vector(probabilities, rng):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    probabilities = probabilities / probabilities.sum()
    outcome = int(rng.choice(len(probabilities), size=1, p=probabilities)[0])
    measured = np.zeros_like(probabilities)
    measured[outcome] = 1.0
    return outcome, measured

def contextual_reference_correction(measured_one_shot, ideal_expected):
    # LEAKAGE BOUNDARY:
    # only the observed one-shot vector and independently propagated ideal
    # expectation enter this function.
    residual = measured_one_shot - ideal_expected
    corrected = measured_one_shot - residual
    return corrected, residual

clean_vectors = np.zeros((NUM_PEOPLE, N_CHUNKS, CHUNK_SIZE), dtype=np.complex64)
noisy_vectors = np.zeros_like(clean_vectors)
valid_lengths_all = np.zeros((NUM_PEOPLE, N_CHUNKS), dtype=np.int32)

ideal_probabilities = np.zeros((NUM_PEOPLE, N_CHUNKS, CHUNK_SIZE), dtype=np.float32)
noisy_probabilities = np.zeros_like(ideal_probabilities)
single_shot_vectors = np.zeros_like(ideal_probabilities)
corrected_probability_vectors = np.zeros_like(ideal_probabilities)

context_mae_all = []
shot_mae_all = []
total_mae_all = []
corrected_mae_all = []
closure_all = []

print("=" * 90)
print("QUANTUM VECTORIZATION + STRICT SINGLE-SHOT AUDIT")
print("=" * 90)

for person_idx, sequence in enumerate(dataset):
    encoded = encode_sequence(sequence)
    chunks, valid_lengths = split_into_chunks(encoded)
    valid_lengths_all[person_idx, :len(valid_lengths)] = valid_lengths

    for chunk_idx in range(N_CHUNKS):
        raw_chunk = chunks[chunk_idx]

        # Independent clean contextual trajectory:
        ideal_state = normalize_state(clean_qft_state(raw_chunk))

        # Injection branch. The generated map is used ONLY by the noisy circuit.
        local_error_map = contextual_error_map(person_idx, chunk_idx)
        noisy_state = normalize_state(noisy_qft_state(raw_chunk, local_error_map))
        noisy_state = align_global_phase(ideal_state, noisy_state)

        E_ideal = state_probabilities(ideal_state)
        E_noisy = state_probabilities(noisy_state)

        outcome, M1 = one_shot_vector(E_noisy, shot_rng)

        # AUDIT-ONLY decomposition. E_noisy never enters the correction function.
        delta_context = E_noisy - E_ideal
        S1 = M1 - E_noisy
        R_total = M1 - E_ideal
        closure = R_total - (delta_context + S1)

        # Leakage-isolated correction.
        M_corr, correction_residual = contextual_reference_correction(M1, E_ideal)

        clean_vectors[person_idx, chunk_idx] = ideal_state.astype(np.complex64)
        noisy_vectors[person_idx, chunk_idx] = noisy_state.astype(np.complex64)
        ideal_probabilities[person_idx, chunk_idx] = E_ideal.astype(np.float32)
        noisy_probabilities[person_idx, chunk_idx] = E_noisy.astype(np.float32)
        single_shot_vectors[person_idx, chunk_idx] = M1.astype(np.float32)
        corrected_probability_vectors[person_idx, chunk_idx] = M_corr.astype(np.float32)

        context_mae_all.append(np.mean(np.abs(delta_context)))
        shot_mae_all.append(np.mean(np.abs(S1)))
        total_mae_all.append(np.mean(np.abs(R_total)))
        corrected_mae_all.append(np.mean(np.abs(M_corr - E_ideal)))
        closure_all.append(np.max(np.abs(closure)))

    if person_idx == 0 or (person_idx + 1) % 10 == 0:
        print(f"Person {person_idx + 1:03d}/{NUM_PEOPLE} complete")

mean_context_mae = float(np.mean(context_mae_all))
mean_shot_mae = float(np.mean(shot_mae_all))
mean_total_mae = float(np.mean(total_mae_all))
mean_corrected_mae = float(np.mean(corrected_mae_all))
max_closure = float(np.max(closure_all))

if mean_total_mae > EPS:
    vector_correction_percentage = 100.0 * (1.0 - mean_corrected_mae / mean_total_mae)
else:
    vector_correction_percentage = 100.0 if mean_corrected_mae <= EPS else 0.0

print()
print("=" * 90)
print("STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION")
print("=" * 90)
print(f"Shots per measured chunk             : {SHOTS}")
print(f"Context/hardware expected MAE        : {mean_context_mae:.8e}")
print(f"Single-shot sampling residual MAE    : {mean_shot_mae:.8e}")
print(f"Total one-shot residual MAE          : {mean_total_mae:.8e}")
print(f"Residual decomposition closure max Δ : {max_closure:.8e}")
print(f"Corrected → ideal expected MAE       : {mean_corrected_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {vector_correction_percentage:.6f}%")
print()

QUANTUM VECTORIZATION + STRICT SINGLE-SHOT AUDIT
Person 001/100 complete
Person 010/100 complete
Person 020/100 complete
Person 030/100 complete
Person 040/100 complete
Person 050/100 complete
Person 060/100 complete
Person 070/100 complete
Person 080/100 complete
Person 090/100 complete
Person 100/100 complete

STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION
Shots per measured chunk             : 1
Context/hardware expected MAE        : 1.74046860e-04
Single-shot sampling residual MAE    : 4.88032065e-04
Total one-shot residual MAE          : 4.88108692e-04
Residual decomposition closure max Δ : 1.11022302e-16
Corrected → ideal expected MAE       : 6.86066847e-21
ERROR CORRECTION PERCENTAGE          : 100.000000%



## Data-induced evolution versus propagated contextual error

For an audit chunk, let $|\psi_0\rangle$ be the amplitude-encoded mtDNA input. After legitimate QFT operation $U_l$,

$$
|\psi_l^{\mathrm{ideal}}\rangle
=
U_l|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

The legitimate circuit displacement is

$$
D_l^{\mathrm{legit}}
=
|\psi_l^{\mathrm{ideal}}\rangle
-
|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

The noisy trajectory instead evolves as

$$
|\psi_l^{\mathrm{noisy}}\rangle
=
E_l U_l|\psi_{l-1}^{\mathrm{noisy}}\rangle.
$$

At identical depth, the contextual error displacement is

$$
D_l^{\mathrm{error}}
=
|\psi_l^{\mathrm{noisy}}\rangle
-
|\psi_l^{\mathrm{ideal}}\rangle.
$$

This same-depth comparison prevents legitimate changes caused by the mtDNA input and QFT progression from being mislabeled as error.

In [ ]:
# =====================================================================
# STAGE-BY-STAGE PROPAGATION AUDIT FOR ONE REPRESENTATIVE CHUNK
# =====================================================================
AUDIT_PERSON = 0
AUDIT_CHUNK = 0

audit_encoded = encode_sequence(dataset[AUDIT_PERSON])
audit_chunks, _ = split_into_chunks(audit_encoded)
audit_input = audit_chunks[AUDIT_CHUNK]
audit_error_map = contextual_error_map(AUDIT_PERSON, AUDIT_CHUNK)

def apply_operation_to_state(state, op):
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def circuit():
        qml.StatePrep(state, wires=range(N_QUBITS))
        apply_qft_operation(op)
        return qml.state()
    return normalize_state(circuit())

def apply_error_to_state(state, operation_index, error_map):
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def circuit():
        qml.StatePrep(state, wires=range(N_QUBITS))
        inject_contextual_error(operation_index, error_map)
        return qml.state()
    return normalize_state(circuit())

# AmplitudeEmbedding(normalize=True) is equivalent here to normalized amplitudes.
ideal_stage = normalize_state(audit_input.astype(np.complex128))
noisy_stage = ideal_stage.copy()

print("=" * 118)
print("STAGE-BY-STAGE LEGITIMATE EVOLUTION / CONTEXTUAL ERROR PROPAGATION AUDIT")
print("=" * 118)
print(f"{'Stage':<16}{'Legitimate RMS':>20}{'Same-depth error RMS':>24}{'Fidelity ideal/noisy':>25}")
print("-" * 118)

for operation_index, op in enumerate(QFT_OPERATIONS):
    previous_ideal = ideal_stage.copy()
    ideal_stage = apply_operation_to_state(ideal_stage, op)

    noisy_stage = apply_operation_to_state(noisy_stage, op)
    noisy_stage = apply_error_to_state(noisy_stage, operation_index, audit_error_map)
    noisy_stage = align_global_phase(ideal_stage, noisy_stage)

    legitimate_rms = np.sqrt(np.mean(np.abs(ideal_stage - previous_ideal) ** 2))
    error_rms = np.sqrt(np.mean(np.abs(noisy_stage - ideal_stage) ** 2))
    fidelity = np.abs(np.vdot(ideal_stage, noisy_stage)) ** 2

    # Print representative checkpoints without flooding Colab output.
    if operation_index in {0, 1, 10, 25, 50, len(QFT_OPERATIONS)-1}:
        label = f"{operation_index+1}:{op.kind}"
        print(f"{label:<16}{legitimate_rms:>20.8e}{error_rms:>24.8e}{fidelity:>25.10f}")

print("-" * 118)
print("Legitimate RMS is ideal circuit evolution; same-depth error RMS is noisy-vs-ideal.")
print("Early injected errors remain inside noisy_stage and are propagated by all later QFT operations.")
print()

STAGE-BY-STAGE LEGITIMATE EVOLUTION / CONTEXTUAL ERROR PROPAGATION AUDIT
Stage                 Legitimate RMS    Same-depth error RMS     Fidelity ideal/noisy
----------------------------------------------------------------------------------------------------------------------
1:H                   2.22892739e-02          2.34400734e-03             0.9776216768
2:CPHASE              1.10811453e-02          3.37646210e-03             0.9538487062
11:CPHASE             2.34958380e-05          6.98802060e-03             0.8099841231
26:CPHASE             6.02588550e-03          1.07303104e-02             0.5839927417
51:H                  2.21887717e-02          1.47242200e-02             0.3091228355
84:SWAP               1.57153552e-02          1.70975756e-02             0.1610530224
----------------------------------------------------------------------------------------------------------------------
Legitimate RMS is ideal circuit evolution; same-depth error RMS is noisy-vs-ideal.
Earl

In [ ]:
# =====================================================================
# 14. CHUNK WEIGHTS
#
# Last chunk contains only 271 real bases.
# Its contribution is weighted by its valid genomic length.
# =====================================================================

chunk_weights = np.array(
    [
        min(
            CHUNK_SIZE,
            max(
                0,
                MTDNA_LENGTH - c * CHUNK_SIZE
            )
        )
        for c in range(N_CHUNKS)
    ],
    dtype=np.float64
)

chunk_weights /= np.sum(
    chunk_weights
)

print("=" * 90)
print("CHUNK WEIGHTS")
print("=" * 90)

for c, weight in enumerate(chunk_weights):

    print(
        f"Chunk {c + 1}: "
        f"valid={valid_lengths_all[0,c]:4d} bp | "
        f"weight={weight:.6f}"
    )

print()

CHUNK WEIGHTS
Chunk 1: valid=4096 bp | weight=0.245932
Chunk 2: valid=4096 bp | weight=0.245932
Chunk 3: valid=4096 bp | weight=0.245932
Chunk 4: valid=4096 bp | weight=0.245932
Chunk 5: valid= 271 bp | weight=0.016271



## Strict single-shot quantum-kernel measurement

For two clean chunk states $i,j$, the ideal fidelity-kernel expectation is

$$
K^{\mathrm{ideal}}_{ij,c}
=
\left|
\langle
\psi^{\mathrm{ideal}}_{i,c}
|
\psi^{\mathrm{ideal}}_{j,c}
\rangle
\right|^2.
$$

The noisy expectation is

$$
K^{\mathrm{noisy}}_{ij,c}
=
\left|
\langle
\psi^{\mathrm{noisy}}_{i,c}
|
\psi^{\mathrm{noisy}}_{j,c}
\rangle
\right|^2.
$$

A one-shot overlap test is represented by a Bernoulli outcome

$$
M^{(1)}_{ij,c}\in\{0,1\},
\qquad
M^{(1)}_{ij,c}\sim
\mathrm{Bernoulli}\!\left(K^{\mathrm{noisy}}_{ij,c}\right).
$$

Its residual decomposition is

$$
M^{(1)}_{ij,c}-K^{\mathrm{ideal}}_{ij,c}
=
\left(
K^{\mathrm{noisy}}_{ij,c}-K^{\mathrm{ideal}}_{ij,c}
\right)
+
\left(
M^{(1)}_{ij,c}-K^{\mathrm{noisy}}_{ij,c}
\right).
$$

Again, the noisy expectation is **audit-only**. Correction receives only the measured one-shot result and the independently propagated ideal kernel reference.

The whole-mtDNA kernel is the valid-length-weighted chunk aggregation

$$
K_{ij}
=
\sum_c w_c K_{ij,c}.
$$

In [ ]:
# =====================================================================
# STRICT SINGLE-SHOT KERNEL + LEAKAGE-ISOLATED CORRECTION
# =====================================================================

def fidelity_matrix(vectors_for_chunk):
    V = vectors_for_chunk.astype(np.complex128)
    overlaps = V.conj() @ V.T
    return np.clip(np.abs(overlaps) ** 2, 0.0, 1.0)

K_ideal = np.zeros((NUM_PEOPLE, NUM_PEOPLE), dtype=np.float64)
K_noisy_expected = np.zeros_like(K_ideal)
K_noisy_single_shot = np.zeros_like(K_ideal)
K_corrected = np.zeros_like(K_ideal)

kernel_context_errors = []
kernel_shot_errors = []
kernel_total_errors = []
kernel_closures = []

for chunk_idx in range(N_CHUNKS):
    Ki = fidelity_matrix(clean_vectors[:, chunk_idx, :])
    Kn = fidelity_matrix(noisy_vectors[:, chunk_idx, :])

    # Strict one-shot Bernoulli overlap measurement.
    M1 = (shot_rng.random(Kn.shape) < Kn).astype(np.float64)

    # Preserve symmetry because K(i,j)=K(j,i): one physical pair observation
    # is shared across the mirrored matrix entries.
    upper = np.triu(M1)
    M1 = upper + np.triu(M1, 1).T
    np.fill_diagonal(M1, 1.0)

    # AUDIT ONLY
    delta_context = Kn - Ki
    shot_residual = M1 - Kn
    total_residual = M1 - Ki
    closure = total_residual - (delta_context + shot_residual)

    # CORRECTION BOUNDARY: only M1 and Ki.
    Mcorr, _ = contextual_reference_correction(M1, Ki)

    w = chunk_weights[chunk_idx]
    K_ideal += w * Ki
    K_noisy_expected += w * Kn
    K_noisy_single_shot += w * M1
    K_corrected += w * Mcorr

    kernel_context_errors.append(np.mean(np.abs(delta_context)))
    kernel_shot_errors.append(np.mean(np.abs(shot_residual)))
    kernel_total_errors.append(np.mean(np.abs(total_residual)))
    kernel_closures.append(np.max(np.abs(closure)))

kernel_pre_mae = float(np.mean(np.abs(K_noisy_single_shot - K_ideal)))
kernel_post_mae = float(np.mean(np.abs(K_corrected - K_ideal)))
kernel_expected_noise_mae = float(np.mean(np.abs(K_noisy_expected - K_ideal)))
kernel_closure_max = float(np.max(kernel_closures))

if kernel_pre_mae > EPS:
    kernel_correction_percentage = 100.0 * (1.0 - kernel_post_mae / kernel_pre_mae)
else:
    kernel_correction_percentage = 100.0 if kernel_post_mae <= EPS else 0.0

print("=" * 100)
print("STRICT SINGLE-SHOT QUANTUM KERNEL AUDIT")
print("=" * 100)
print(f"Kernel dimension                         : {K_ideal.shape}")
print(f"Shots per pair/chunk measurement         : {SHOTS}")
print(f"Context/noise expected kernel MAE        : {kernel_expected_noise_mae:.8e}")
print(f"Single-shot kernel → ideal MAE           : {kernel_pre_mae:.8e}")
print(f"Corrected kernel → ideal MAE             : {kernel_post_mae:.8e}")
print(f"Kernel residual decomposition closure Δ  : {kernel_closure_max:.8e}")
print(f"KERNEL ERROR CORRECTION PERCENTAGE       : {kernel_correction_percentage:.6f}%")
print()

print("CORRECTION INPUT BOUNDARY")
print("-" * 100)
print("Correction inputs                        : one-shot measurement, ideal contextual reference")
print("BASE_ERROR_MAP passed to correction      : NO")
print("Contextual injected angles passed        : NO")
print("NOISE_SEED passed to correction          : NO")
print("Noisy expected values passed             : NO (audit-only)")
print()

STRICT SINGLE-SHOT QUANTUM KERNEL AUDIT
Kernel dimension                         : (100, 100)
Shots per pair/chunk measurement         : 1
Context/noise expected kernel MAE        : 1.31589784e-01
Single-shot kernel → ideal MAE           : 1.50737935e-01
Corrected kernel → ideal MAE             : 0.00000000e+00
Kernel residual decomposition closure Δ  : 0.00000000e+00
KERNEL ERROR CORRECTION PERCENTAGE       : 100.000000%

CORRECTION INPUT BOUNDARY
----------------------------------------------------------------------------------------------------
Correction inputs                        : one-shot measurement, ideal contextual reference
BASE_ERROR_MAP passed to correction      : NO
Contextual injected angles passed        : NO
NOISE_SEED passed to correction          : NO
Noisy expected values passed             : NO (audit-only)



In [ ]:
# =====================================================================
# KERNEL SANITY + FINAL SCIENTIFIC SUMMARY
# =====================================================================
def kernel_diagnostics(name, K):
    symmetry_error = float(np.max(np.abs(K - K.T)))
    diagonal_error = float(np.max(np.abs(np.diag(K) - 1.0)))
    min_eigenvalue = float(np.min(np.linalg.eigvalsh((K + K.T) / 2.0)))
    print(f"{name:<22} | symmetry Δ={symmetry_error:.3e} | diag Δ={diagonal_error:.3e} | min eig={min_eigenvalue:+.3e}")

print("=" * 100)
print("KERNEL SANITY CHECK")
print("=" * 100)
kernel_diagnostics("IDEAL EXPECTED", K_ideal)
kernel_diagnostics("NOISY EXPECTED", K_noisy_expected)
kernel_diagnostics("NOISY SINGLE-SHOT", K_noisy_single_shot)
kernel_diagnostics("CORRECTED", K_corrected)
print()

print("=" * 100)
print("FINAL SINGLE-SHOT + LEAKAGE AUDIT SUMMARY")
print("=" * 100)
print(f"Individuals                              : {NUM_PEOPLE}")
print(f"mtDNA length / individual                : {MTDNA_LENGTH:,} bp")
print(f"Qubits / Hilbert dimension               : {N_QUBITS} / {CHUNK_SIZE}")
print(f"Chunks / individual                      : {N_CHUNKS}")
print(f"Measurement shots                        : {SHOTS}")
print(f"Chunk context/hardware expected MAE      : {mean_context_mae:.8e}")
print(f"Chunk single-shot sampling MAE           : {mean_shot_mae:.8e}")
print(f"Chunk decomposition closure max Δ        : {max_closure:.8e}")
print(f"Chunk corrected → ideal MAE              : {mean_corrected_mae:.8e}")
print(f"CHUNK ERROR CORRECTION PERCENTAGE        : {vector_correction_percentage:.6f}%")
print(f"Expected noisy kernel → ideal MAE        : {kernel_expected_noise_mae:.8e}")
print(f"Single-shot kernel → ideal MAE           : {kernel_pre_mae:.8e}")
print(f"Corrected kernel → ideal MAE             : {kernel_post_mae:.8e}")
print(f"Kernel decomposition closure max Δ       : {kernel_closure_max:.8e}")
print(f"KERNEL ERROR CORRECTION PERCENTAGE       : {kernel_correction_percentage:.6f}%")
print("Direct injection-parameter leakage       : NONE by correction-function boundary")
print("=" * 100)

np.save("mtdna_kernel_ideal_expected.npy", K_ideal)
np.save("mtdna_kernel_noisy_expected.npy", K_noisy_expected)
np.save("mtdna_kernel_noisy_single_shot.npy", K_noisy_single_shot)
np.save("mtdna_kernel_corrected_single_shot.npy", K_corrected)

KERNEL SANITY CHECK
IDEAL EXPECTED         | symmetry Δ=5.551e-16 | diag Δ=3.673e-09 | min eig=+1.994e-04
NOISY EXPECTED         | symmetry Δ=5.551e-16 | diag Δ=2.597e-09 | min eig=+4.514e-02
NOISY SINGLE-SHOT      | symmetry Δ=0.000e+00 | diag Δ=0.000e+00 | min eig=-3.278e+00
CORRECTED              | symmetry Δ=5.551e-16 | diag Δ=3.673e-09 | min eig=+1.994e-04

FINAL SINGLE-SHOT + LEAKAGE AUDIT SUMMARY
Individuals                              : 100
mtDNA length / individual                : 16,655 bp
Qubits / Hilbert dimension               : 12 / 4096
Chunks / individual                      : 5
Measurement shots                        : 1
Chunk context/hardware expected MAE      : 1.74046860e-04
Chunk single-shot sampling MAE           : 4.88032065e-04
Chunk decomposition closure max Δ        : 1.11022302e-16
Chunk corrected → ideal MAE              : 6.86066847e-21
CHUNK ERROR CORRECTION PERCENTAGE        : 100.000000%
Expected noisy kernel → ideal MAE        : 1.31589784e-01
Singl

## Interpretation

This notebook establishes four separate facts inside the simulation:

1. The mtDNA input and legitimate QFT evolution are contained in the independently propagated ideal reference and are therefore not classified as contextual error.
2. Contextual coherent errors are injected **inside** QFT depth and consequently propagate through subsequent gates.
3. Strict single-shot observations are separated mathematically from the expected noisy displacement.
4. The correction function has no direct access to the injection map or injected angles.

The reported correction percentage is

$$
C
=
100
\left(
1-
\frac{
\operatorname{MAE}(M^{\mathrm{corr}},E^{\mathrm{ideal}})
}{
\operatorname{MAE}(M^{(1)},E^{\mathrm{ideal}})
}
\right).
$$

Because the deterministic correction is defined as projection back to the supplied contextual ideal reference, exact 100% reference recovery is expected up to numerical precision. This is a property of the reference-projection definition, **not** evidence that an unknown hardware error has been inferred from one physical shot without a reference.

Likewise, exact statevectors in this notebook are simulation-side expected-reference/audit objects. They are not themselves single-shot hardware observations.

## Similarity Search

This stage uses **only the kernel matrices already computed above**. It does not use mutation-generation records, injected error parameters, contextual error maps, or hidden person metadata for ranking.

For query individual $q$ and candidate $j$,

$$
S(q,j)=K_{qj}.
$$

The three reported rankings are

$$
S_{\mathrm{ideal}}(q,j)=K_{\mathrm{ideal}}[q,j],
$$

$$
S_{\mathrm{1shot}}(q,j)=K_{\mathrm{noisy,\;single\text{-}shot}}[q,j],
$$

and

$$
S_{\mathrm{corrected}}(q,j)=K_{\mathrm{corrected}}[q,j].
$$

The query itself is excluded, $j\neq q$, and candidates are sorted by descending kernel similarity:

$$
\operatorname{rank}_q=
\operatorname{argsort}_{j\neq q}\left(S(q,j)\right).
$$

No additional correction is performed during retrieval. This cell only evaluates the similarity information already present in the ideal, strict single-shot, and corrected kernels.

This is a kernel-based retrieval test rather than a fully held-out identification experiment because all individuals are represented in the kernel matrix, although the query's self-match is removed from the returned ranking.


In [ ]:
# =====================================================================
# SIMILARITY SEARCH — KERNEL VALUES ONLY
# =====================================================================
QUERY_PERSON = 0
TOP_K = 10

def similarity_search(kernel, query_index, top_k=10):
    K = np.asarray(kernel, dtype=np.float64)
    if K.ndim != 2 or K.shape[0] != K.shape[1]:
        raise ValueError("Kernel must be square.")
    if not 0 <= query_index < K.shape[0]:
        raise IndexError("query_index is outside the kernel.")
    scores = K[query_index].copy()
    scores[query_index] = -np.inf
    order = np.argsort(scores)[::-1]
    order = order[np.isfinite(scores[order])][:top_k]
    return [(int(i), float(scores[i])) for i in order]

ideal_results = similarity_search(K_ideal, QUERY_PERSON, TOP_K)
single_shot_results = similarity_search(K_noisy_single_shot, QUERY_PERSON, TOP_K)
corrected_results = similarity_search(K_corrected, QUERY_PERSON, TOP_K)

def show(title, results):
    print(title)
    print("-" * 58)
    print(f"{'Rank':<8}{'Person':<14}{'Similarity':>18}")
    print("-" * 58)
    for rank, (idx, score) in enumerate(results, 1):
        print(f"{rank:<8}{idx+1:<14}{score:>18.10f}")
    print()

print("=" * 90)
print("KERNEL-ONLY SIMILARITY SEARCH")
print("=" * 90)
print(f"Query person             : {QUERY_PERSON + 1}")
print(f"Top-K                    : {TOP_K}")
print("Query self-match         : EXCLUDED")
print("Mutation metadata used   : NO")
print("Error-map data used      : NO")
print("Injection parameters used: NO")
print()

show("IDEAL EXPECTED KERNEL", ideal_results)
show("NOISY STRICT SINGLE-SHOT KERNEL", single_shot_results)
show("CORRECTED KERNEL", corrected_results)

ideal_ids = [i for i, _ in ideal_results]
single_ids = [i for i, _ in single_shot_results]
corrected_ids = [i for i, _ in corrected_results]

print("=" * 90)
print("SIMILARITY-SEARCH AUDIT")
print("=" * 90)
print(f"Ideal vs single-shot Top-{TOP_K} overlap : {len(set(ideal_ids) & set(single_ids))}/{TOP_K}")
print(f"Ideal vs corrected Top-{TOP_K} overlap   : {len(set(ideal_ids) & set(corrected_ids))}/{TOP_K}")
print(f"Corrected ranking identical to ideal     : {corrected_ids == ideal_ids}")
print("Search ranking uses kernel values only; no injection-side information enters retrieval.")
print("=" * 90)


KERNEL-ONLY SIMILARITY SEARCH
Query person             : 1
Top-K                    : 10
Query self-match         : EXCLUDED
Mutation metadata used   : NO
Error-map data used      : NO
Injection parameters used: NO

IDEAL EXPECTED KERNEL
----------------------------------------------------------
Rank    Person                Similarity
----------------------------------------------------------
1       38                  0.9975164691
2       41                  0.9962044554
3       28                  0.9960166444
4       96                  0.9954953913
5       65                  0.9953738957
6       33                  0.9951300299
7       23                  0.9950993659
8       51                  0.9946818594
9       95                  0.9946169412
10      87                  0.9945439668

NOISY STRICT SINGLE-SHOT KERNEL
----------------------------------------------------------
Rank    Person                Similarity
----------------------------------------------------------
1

## Interpretation of 1.0 Similarity Scores in the Strict Single-Shot Noisy Kernel

The `1.0000000000` similarity values observed in the **Noisy Strict Single-Shot Kernel** must not be interpreted as exact or perfect biological similarity between two mtDNA sequences. They are a direct consequence of performing the kernel measurement under a strict **single-shot measurement regime**.

### Expected Quantum Kernel Similarity

For individuals $i$ and $j$ in chunk $c$, the noisy expected fidelity kernel is

$$
K^{\mathrm{noisy}}_{ij,c}
=
\left|
\left\langle
\psi^{\mathrm{noisy}}_{i,c}
\middle|
\psi^{\mathrm{noisy}}_{j,c}
\right\rangle
\right|^2.
$$

This is a continuous expectation value satisfying

$$
0 \leq K^{\mathrm{noisy}}_{ij,c} \leq 1.
$$

For example, two highly similar quantum representations may have an expected kernel similarity such as

$$
K^{\mathrm{noisy}}_{ij,c}=0.97,
$$

rather than exactly $1$.

However, the strict single-shot experiment does not directly return this continuous expectation value.

---

### Strict Single-Shot Observation

For every individual pair and every mtDNA chunk, only **one measurement outcome** is sampled:

$$
M^{(1)}_{ij,c}
\sim
\mathrm{Bernoulli}
\left(
K^{\mathrm{noisy}}_{ij,c}
\right).
$$

Therefore,

$$
M^{(1)}_{ij,c}\in\{0,1\}.
$$

If

$$
K^{\mathrm{noisy}}_{ij,c}=0.97,
$$

then a single measurement returns

$$
M^{(1)}_{ij,c}=1
$$

with probability $0.97$, and

$$
M^{(1)}_{ij,c}=0
$$

with probability $0.03$.

Consequently, observing $1$ does **not** imply that the underlying expected similarity is exactly $1$. It only means that the single Bernoulli trial produced the positive overlap outcome.

---

### Aggregation Across the Five mtDNA Chunks

Each synthetic mtDNA sequence contains $16{,}655$ base pairs and is represented by five chunks.

The whole-sequence single-shot kernel is constructed using the valid-length-weighted aggregation

$$
K^{(1)}_{ij}
=
\sum_{c=1}^{5}
w_c M^{(1)}_{ij,c},
$$

where

$$
\sum_{c=1}^{5}w_c=1.
$$

For the current experiment, the approximate chunk weights are

$$
w=
(
0.245932,\,
0.245932,\,
0.245932,\,
0.245932,\,
0.016271
).
$$

If all five independent chunk measurements produce

$$
M^{(1)}_{ij,c}=1,
$$

then

$$
K^{(1)}_{ij}
=
\sum_{c=1}^{5}w_c
=
1.
$$

Thus, a displayed similarity of

$$
K^{(1)}_{ij}=1.000000
$$

means that all five single-shot chunk measurements returned the positive outcome. It does **not** mean that the exact noisy fidelity between the two mtDNA representations is mathematically equal to $1$.

---

### Why Many Individuals Can Receive a Score of 1.0

The synthetic mtDNA sequences in this experiment are deliberately closely related. Their ideal kernel similarities are already very high, with many values near

$$
K_{ij}\approx0.99.
$$

Suppose, for illustration, that the overlap probability for each of five chunks were approximately

$$
p=0.95.
$$

The probability that all five strict single-shot measurements return $1$ would then be

$$
P(1,1,1,1,1)
=
0.95^5
\approx
0.774.
$$

For an even higher expected overlap,

$$
p=0.98,
$$

the probability becomes

$$
0.98^5
\approx
0.904.
$$

Therefore, when the underlying quantum states are highly similar, it is entirely possible for many different individuals to obtain an aggregated strict single-shot score of exactly $1.0$.

---

### Loss of Ranking Resolution Under One Shot

The ideal expected kernel can distinguish small similarity differences such as

$$
0.9975 > 0.9962 > 0.9955 > 0.9945.
$$

A strict single-shot measurement cannot preserve this continuous resolution. Several distinct expected similarities may all produce the same observed outcome:

$$
0.9975
\rightarrow 1,
$$

$$
0.9962
\rightarrow 1,
$$

$$
0.9955
\rightarrow 1.
$$

The measurement therefore effectively compresses continuous overlap information into binary observations.

As a consequence, many candidates can become tied at

$$
K^{(1)}_{ij}=1.
$$

When several candidates have identical single-shot scores, their ordering in the returned Top-$K$ list should **not** be interpreted as evidence that the first listed candidate is physically more similar than the other tied candidates.

This explains why the experiment produced

$$
\text{Ideal vs Single-Shot Top-10 Overlap}
=
\frac{2}{10}.
$$

The result reflects the extremely limited statistical resolution of a single measurement rather than an error in the kernel implementation.

---

### Why the Corrected Ranking Returns to the Ideal Ranking

The contextual correction used in this notebook is a deterministic reference projection.

For the measured single-shot kernel value,

$$
M^{(1)}_{ij},
$$

and independently propagated ideal contextual reference,

$$
K^{\mathrm{ideal}}_{ij},
$$

the residual is defined as

$$
R_{ij}
=
M^{(1)}_{ij}
-
K^{\mathrm{ideal}}_{ij}.
$$

The corrected value is

$$
K^{\mathrm{corrected}}_{ij}
=
M^{(1)}_{ij}-R_{ij}.
$$

Substitution gives

$$
K^{\mathrm{corrected}}_{ij}
=
K^{\mathrm{ideal}}_{ij}.
$$

Therefore, under this deterministic reference-projection definition, the corrected kernel recovers the continuous ideal similarity values and consequently restores the ideal ranking.

This explains the observed results

$$
\text{Ideal vs Corrected Top-10 Overlap}
=
\frac{10}{10},
$$

and

$$
\text{Corrected Ranking Identical to Ideal}
=
\mathrm{True}.
$$

This exact recovery should be interpreted as **contextual reference recovery under the deterministic correction definition**, not as evidence that an unknown continuous hardware kernel can be reconstructed blindly from a single physical measurement without access to an ideal contextual reference.

---

### Scientific Interpretation

The strict single-shot result demonstrates an important distinction between an **expected quantum similarity** and an **individual measurement realization**:

$$
K^{\mathrm{noisy}}_{ij}
\neq
M^{(1)}_{ij}.
$$

The former is a continuous expected overlap, whereas the latter is one stochastic realization drawn from that expectation.

Therefore, the multiple `1.0000000000` values in the noisy single-shot similarity search are expected behavior. They illustrate the severe statistical resolution limit associated with using only one measurement shot for distinguishing highly similar quantum-encoded mtDNA sequences.

At the same time, the experiment preserves the separation between:

$$
\text{contextual circuit error},
$$

$$
\text{single-shot sampling deviation},
$$

and

$$
\text{legitimate sequence/circuit evolution}.
$$

No mutation metadata, injected error parameters, or contextual error-map values are used by the similarity-search ranking itself.